# Comparación de datasets — Docentes 2025 vs 2026

Corre las funciones de `compare_datasets_generic.py` sobre los datasets de **docentes**:

- `reporte_comparacion` / `compare_datasets` → diferencias de columnas, categorías y cantidades entre 2025 y 2026.
- `reporte_duplicados` / `detectar_duplicados` → filas repetidas en cada año.

Por defecto usa los **CSV limpios** generados por `estructura.py`. Para comparar los `.xlsx` crudos, cambiá `USAR_LIMPIOS = False` en la celda de config.

In [20]:
from pathlib import Path

import pandas as pd

# Todas las funciones vienen centralizadas del modulo compare_datasets_generic.py
from compare_datasets_generic import (
    cargar,                 # lee csv/xlsx segun extension
    preparar_para_comparar, # unifica dtypes para poder comparar
    compare_datasets,       # reporte estructurado (dict)
    reporte_comparacion,    # reporte en texto
    detectar_desfasajes,    # deteccion operable (con_filas=True -> agrega n_filas)
    detectar_duplicados,    # mascara de duplicados
    reporte_duplicados,     # reporte de duplicados en texto
    limpiar_duplicados_df,  # limpieza -> DataFrame nuevo sin duplicados
)

In [21]:
# ---- Config ----
USAR_LIMPIOS = True  # True: CSV limpios de processed/ ; False: .xlsx crudos

# Carpeta fija: solo los datos de 2025-2025.
CARPETA_DATOS = Path.home() / "Downloads" / "Datos Ceibal 2025-2025"
assert CARPETA_DATOS.exists(), f"No existe la carpeta de datos: {CARPETA_DATOS}"

if USAR_LIMPIOS:
    RUTA_2025 = CARPETA_DATOS / "processed" / "docentes_2025_clean.csv"
    RUTA_2026 = CARPETA_DATOS / "processed" / "docentes_2026_clean.csv"
else:
    RUTA_2025 = CARPETA_DATOS / "datosUCU2025_doc.xlsx"
    RUTA_2026 = CARPETA_DATOS / "datosUCU2026_doc.xlsx"

print("Carpeta:", CARPETA_DATOS)
print("2025   :", RUTA_2025.name)
print("2026   :", RUTA_2026.name)

Carpeta: /Users/gustavohunter/Downloads/Datos Ceibal 2025-2025
2025   : docentes_2025_clean.csv
2026   : docentes_2026_clean.csv


In [22]:
doc25 = cargar(RUTA_2025)
doc26 = cargar(RUTA_2026)

print(f"docentes 2025: {doc25.shape[0]:,} filas x {doc25.shape[1]} cols")
print(f"docentes 2026: {doc26.shape[0]:,} filas x {doc26.shape[1]} cols")
doc25.head()

docentes 2025: 148,372 filas x 25 cols
docentes 2026: 155,745 filas x 25 cols


,ID_persona,ID_CENTRO_docentes,Dias6,Dias5,Dias4,Sexo,ZONA,tipo_centro,Rubro,dept_nombre,...,IVSMEDIA,ANIOLECTIVO,ivsmedia_q,contexto_q,contexto_zona,vuln_q,dias_totales,accedio,tipo,anio
0,ID_00001,CENTRO_0001,14,14,12,F,Urbana,Liceo Público,DGES,MONTEVIDEO,...,NaN,2025,NaN,NaN,NaN,NaN,40,1,docentes,2025
1,ID_00001,CENTRO_0002,14,14,12,F,Urbana,Liceo Público,DGES,MONTEVIDEO,...,NaN,2025,NaN,NaN,NaN,NaN,40,1,docentes,2025
2,ID_00002,CENTRO_0003,1,0,0,M,Urbana,Liceo Público,DGES,CANELONES,...,NaN,2025,NaN,NaN,NaN,NaN,1,1,docentes,2025
3,ID_00003,CENTRO_0004,2,6,11,F,Urbana,Liceo Público,DGES,SORIANO,...,NaN,2025,NaN,NaN,NaN,NaN,19,1,docentes,2025
4,ID_00003,CENTRO_0004,2,6,11,F,Urbana,Liceo Público,DGES,SORIANO,...,NaN,2025,NaN,NaN,NaN,NaN,19,1,docentes,2025


### Preparación para comparar

`compare_datasets` agrupa y hace un `merge` interno sobre las columnas de texto. Dos situaciones de estos datos lo rompen y `preparar_para_comparar` (del módulo) las resuelve antes:

1. **Columnas vacías en un solo año** — p. ej. `IVSMEDIA` está 100% vacía en 2025 (docentes) y `contexto_zona` está vacía en 2026. Una columna toda-NaN se lee como `float64` y no aporta comparación de valores → **se excluyen**.
2. **Columnas numéricas en un año y texto en el otro** (con datos en ambos) → se pasan a **texto** en los dos para poder compararlas como categorías.

In [23]:
doc25c, doc26c = preparar_para_comparar(doc25, doc26)

Columnas vacias en un dataset (excluidas): ['IVSMEDIA', 'ivsmedia_q', 'contexto_zona']
Columnas mixto (num/texto) pasadas a texto: (ninguna)


## 1. Reporte de comparación (texto legible)

`reporte_comparacion` corre internamente `compare_datasets` y arma un texto listo para leer.

In [24]:

print(reporte_comparacion(doc25c, doc26c, name1="2025", name2="2026"))

COMPARACIÓN: 2025  vs  2026
Filas totales -> 2025: 148372 | 2026: 155745
Columnas categóricas comparadas: ['ID_persona', 'ID_CENTRO_docentes', 'Sexo', 'ZONA', 'tipo_centro', 'Rubro', 'dept_nombre', 'ciclo', 'grado', 'grupo', 'MATERIA', 'CONTEXTO', 'tipo']
Columnas numéricas comparadas:   ['Dias6', 'Dias5', 'Dias4', 'ANIOLECTIVO', 'contexto_q', 'vuln_q', 'dias_totales', 'accedio', 'anio']

1) COLUMNAS QUE NO COINCIDEN
------------------------------------------------------------
Solo en 2025:
  (ninguna)
Solo en 2026:
  (ninguna)

2) CATEGORÍAS QUE NO COINCIDEN
------------------------------------------------------------
Columna 'ID_persona':
  Solo en 2025:
    - id_00001
    - id_00002
    - id_00005
    - id_00008
    - id_00010
    - id_00014
    - id_00019
    - id_00020
    - id_00021
    - id_00023
    - id_00031
    - id_00032
    - id_00034
    - id_00039
    - id_00047
    - id_00064
    - id_00066
    - id_00069
    - id_00075
    - id_00090
    ... y 5317 más
  Solo en 2026:


## 2. Resultado estructurado (dict con DataFrames)

`compare_datasets` devuelve el detalle completo para inspeccionar a mano.

In [25]:
resultado = compare_datasets(doc25c, doc26c, name1="2025", name2="2026")
resultado["resumen"]

{'columnas_solo_en_2025': [],
 'columnas_solo_en_2026': [],
 'columnas_categoricas_comparadas': ['ID_persona',
  'ID_CENTRO_docentes',
  'Sexo',
  'ZONA',
  'tipo_centro',
  'Rubro',
  'dept_nombre',
  'ciclo',
  'grado',
  'grupo',
  'MATERIA',
  'CONTEXTO',
  'tipo'],
 'columnas_numericas_comparadas': ['Dias6',
  'Dias5',
  'Dias4',
  'ANIOLECTIVO',
  'contexto_q',
  'vuln_q',
  'dias_totales',
  'accedio',
  'anio'],
 'columnas_con_categorias_desfasadas': ['ID_persona',
  'ID_CENTRO_docentes',
  'ciclo',
  'grado',
  'grupo',
  'MATERIA'],
 'combinaciones_comunes_comparadas': 25907,
 'combinaciones_con_cantidad_desfasada': 160817,
 'filas_totales_2025': 148372,
 'filas_totales_2026': 155745}

In [26]:
# Categorias (valores de texto) que aparecen en un solo anio
resultado["categorias_desfasadas"]

{'ID_persona': {'solo_en_2025': ['id_00001',
   'id_00002',
   'id_00005',
   'id_00008',
   'id_00010',
   'id_00014',
   'id_00019',
   'id_00020',
   'id_00021',
   'id_00023',
   'id_00031',
   'id_00032',
   'id_00034',
   'id_00039',
   'id_00047',
   'id_00064',
   'id_00066',
   'id_00069',
   'id_00075',
   'id_00090',
   'id_00093',
   'id_00100',
   'id_00117',
   'id_00119',
   'id_00122',
   'id_00125',
   'id_00127',
   'id_00132',
   'id_00135',
   'id_00139',
   'id_00142',
   'id_00156',
   'id_00164',
   'id_00167',
   'id_00183',
   'id_00195',
   'id_00202',
   'id_00206',
   'id_00211',
   'id_00213',
   'id_00214',
   'id_00218',
   'id_00231',
   'id_00234',
   'id_00236',
   'id_00244',
   'id_00252',
   'id_00256',
   'id_00266',
   'id_00295',
   'id_00304',
   'id_00315',
   'id_00319',
   'id_00325',
   'id_00326',
   'id_00334',
   'id_00335',
   'id_00337',
   'id_00338',
   'id_00345',
   'id_00350',
   'id_00352',
   'id_00357',
   'id_00371',
   'id_003

In [27]:
# Combinaciones de categorias con cantidades numericas distintas entre anios
resultado["cantidades_desfasadas"].head(20)

,ID_persona,ID_CENTRO_docentes,Sexo,ZONA,tipo_centro,Rubro,dept_nombre,ciclo,grado,grupo,MATERIA,CONTEXTO,tipo,columna,valor_2025,valor_2026,diferencia
0,id_00011,centro_0014,m,urbana,liceo publico,dges,rivera,4to. ciclo,2,2do.csh g. 3,filosofia,NaN,docentes,Dias6,17.0,14.0,3.0
1,id_00011,centro_0014,m,urbana,liceo publico,dges,rivera,4to. ciclo,2,2do.csh g. 5,filosofia,NaN,docentes,Dias6,17.0,14.0,3.0
2,id_00011,centro_0014,m,urbana,liceo publico,dges,rivera,bachillerato,2,2do.hum. g. 1_1osem,filosofia,NaN,docentes,Dias6,17.0,14.0,3.0
3,id_00013,centro_0017,f,urbana,utu,dgetp,montevideo,educacion basica integrada,7,7a||0,matematica,NaN,docentes,Dias6,6.0,2.0,4.0
4,id_00013,centro_0017,f,urbana,utu,dgetp,montevideo,educacion basica integrada,7,7b||0,matematica,NaN,docentes,Dias6,6.0,2.0,4.0
5,id_00013,centro_0017,f,urbana,utu,dgetp,montevideo,educacion basica integrada,8,8a||0,matematica,NaN,docentes,Dias6,6.0,2.0,4.0
6,id_00013,centro_0017,f,urbana,utu,dgetp,montevideo,educacion basica integrada,8,8b||0,matematica,NaN,docentes,Dias6,6.0,2.0,4.0
7,id_00016,centro_0020,m,urbana,liceo publico,dges,montevideo,bachillerato,2,2do.hum. g. 1_1osem,historia,NaN,docentes,Dias6,1.0,0.0,1.0
8,id_00017,centro_0022,m,urbana,liceo publico,dges,canelones,ciclo basico,1,1ro.ee g. 2_anual_s1,informatica e.e.,NaN,docentes,Dias6,21.0,22.0,-1.0
9,id_00017,centro_0022,m,urbana,liceo publico,dges,canelones,ciclo basico,2,2do.ee g. 2_anual_s1,informatica e.e.,NaN,docentes,Dias6,21.0,22.0,-1.0


## 3. Duplicados por año

`reporte_duplicados` informa (no modifica); `detectar_duplicados` devuelve la máscara operable.

In [28]:
print(reporte_duplicados(doc25))
print()
print(reporte_duplicados(doc26))

DUPLICADOS
Filas totales: 148372
Columnas consideradas: todas
Grupos de valores repetidos: 6205
Filas involucradas en duplicados (incluye la primera copia): 12654
Filas 'extra' (sobrarían si se deduplica dejando 1 copia por grupo): 6449
Porcentaje de filas extra sobre el total: 4.3%

DUPLICADOS
Filas totales: 155745
Columnas consideradas: todas
Grupos de valores repetidos: 553
Filas involucradas en duplicados (incluye la primera copia): 1194
Filas 'extra' (sobrarían si se deduplica dejando 1 copia por grupo): 641
Porcentaje de filas extra sobre el total: 0.4%


In [29]:
# Ejemplo: ver las filas duplicadas de 2025 (sin eliminar nada)
mask_dup_25 = detectar_duplicados(doc25, keep=False)
doc25[mask_dup_25].head(20)

,ID_persona,ID_CENTRO_docentes,Dias6,Dias5,Dias4,Sexo,ZONA,tipo_centro,Rubro,dept_nombre,...,IVSMEDIA,ANIOLECTIVO,ivsmedia_q,contexto_q,contexto_zona,vuln_q,dias_totales,accedio,tipo,anio
16,ID_00005,CENTRO_0006,13,16,13,F,Urbana,Escuela Pública,DGEIP,MONTEVIDEO,...,NaN,2025,NaN,5.0,Urbano,5.0,42,1,docentes,2025
17,ID_00005,CENTRO_0006,13,16,13,F,Urbana,Escuela Pública,DGEIP,MONTEVIDEO,...,NaN,2025,NaN,5.0,Urbano,5.0,42,1,docentes,2025
18,ID_00006,CENTRO_0007,14,18,11,F,Urbana,Escuela Pública,DGEIP,MONTEVIDEO,...,NaN,2025,NaN,1.0,Urbano,1.0,43,1,docentes,2025
19,ID_00006,CENTRO_0007,14,18,11,F,Urbana,Escuela Pública,DGEIP,MONTEVIDEO,...,NaN,2025,NaN,1.0,Urbano,1.0,43,1,docentes,2025
74,ID_00020,CENTRO_0025,3,9,14,F,Urbana,Escuela Pública,DGEIP,MONTEVIDEO,...,NaN,2025,NaN,4.0,Urbano,4.0,26,1,docentes,2025
75,ID_00020,CENTRO_0025,3,9,14,F,Urbana,Escuela Pública,DGEIP,MONTEVIDEO,...,NaN,2025,NaN,4.0,Urbano,4.0,26,1,docentes,2025
85,ID_00023,CENTRO_0029,2,10,5,F,Urbana,Escuela Pública,DGEIP,MONTEVIDEO,...,NaN,2025,NaN,4.0,Urbano,4.0,17,1,docentes,2025
86,ID_00023,CENTRO_0029,2,10,5,F,Urbana,Escuela Pública,DGEIP,MONTEVIDEO,...,NaN,2025,NaN,4.0,Urbano,4.0,17,1,docentes,2025
149,ID_00038,CENTRO_0049,8,15,6,F,Urbana,Liceo Público,DGES,MONTEVIDEO,...,NaN,2025,NaN,NaN,NaN,NaN,29,1,docentes,2025
150,ID_00038,CENTRO_0049,8,15,6,F,Urbana,Liceo Público,DGES,MONTEVIDEO,...,NaN,2025,NaN,NaN,NaN,NaN,29,1,docentes,2025


## 4. Detección y limpieza (centralizadas en el módulo)

`compare_datasets_generic.py` completa el patrón *reporte → detección → limpieza* para cada tema. El notebook solo **importa** y usa:

| Tema | Reporte | Detección (operable) | Limpieza → DataFrame nuevo |
|---|---|---|---|
| Comparación | `reporte_comparacion` / `compare_datasets` | `detectar_desfasajes` | — |
| Duplicados | `reporte_duplicados` | `detectar_duplicados` | `limpiar_duplicados_df` |

### 4a. Comparación → detección operable

`detectar_desfasajes` deriva de `compare_datasets` (que solo reporta) y devuelve tablas listas para filtrar/contar, igual que `detectar_duplicados` devuelve una máscara. Con `con_filas=True` retorna además la columna `n_filas`.

In [30]:
# Detección operable (tabla) a partir del reporte de comparación
desfasajes = detectar_desfasajes(doc25c, doc26c, name1="2025", name2="2026")

print("Categorias desfasadas (filas):", desfasajes["categorias"].shape[0])
print("Cantidades desfasadas (filas):", desfasajes["cantidades"].shape[0])

# Cuántos valores desfasados hay por columna y en qué año aparecen
resumen_cat = (desfasajes["categorias"]
               .groupby(["columna", "solo_en"]).size()
               .unstack(fill_value=0))
resumen_cat

Categorias desfasadas (filas): 12971
Cantidades desfasadas (filas): 160817


solo_en,2025,2026
columna,,
ID_CENTRO_docentes,55,136
ID_persona,5337,6916
MATERIA,126,185
ciclo,2,2
grado,0,1
grupo,109,102


### 4b. Duplicados → limpieza que devuelve un DataFrame nuevo

`detectar_duplicados` devuelve la máscara operable; `limpiar_duplicados_df` (del módulo) la aplica y retorna un **DataFrame nuevo** sin las filas duplicadas, sin modificar el original. Se aplica a cada año y se guarda como `docentes_<anio>_dedup.csv`.

In [31]:
print("Docentes 2025:")
doc25_sin_dups = limpiar_duplicados_df(doc25, verbose=True)
print("Docentes 2026:")
doc26_sin_dups = limpiar_duplicados_df(doc26, verbose=True)
doc25_sin_dups.head()

Docentes 2025:
  6449 filas duplicadas eliminadas: 148372 -> 141923
Docentes 2026:
  641 filas duplicadas eliminadas: 155745 -> 155104


,ID_persona,ID_CENTRO_docentes,Dias6,Dias5,Dias4,Sexo,ZONA,tipo_centro,Rubro,dept_nombre,...,IVSMEDIA,ANIOLECTIVO,ivsmedia_q,contexto_q,contexto_zona,vuln_q,dias_totales,accedio,tipo,anio
0,ID_00001,CENTRO_0001,14,14,12,F,Urbana,Liceo Público,DGES,MONTEVIDEO,...,NaN,2025,NaN,NaN,NaN,NaN,40,1,docentes,2025
1,ID_00001,CENTRO_0002,14,14,12,F,Urbana,Liceo Público,DGES,MONTEVIDEO,...,NaN,2025,NaN,NaN,NaN,NaN,40,1,docentes,2025
2,ID_00002,CENTRO_0003,1,0,0,M,Urbana,Liceo Público,DGES,CANELONES,...,NaN,2025,NaN,NaN,NaN,NaN,1,1,docentes,2025
3,ID_00003,CENTRO_0004,2,6,11,F,Urbana,Liceo Público,DGES,SORIANO,...,NaN,2025,NaN,NaN,NaN,NaN,19,1,docentes,2025
4,ID_00003,CENTRO_0004,2,6,11,F,Urbana,Liceo Público,DGES,SORIANO,...,NaN,2025,NaN,NaN,NaN,NaN,19,1,docentes,2025


In [32]:
# Guarda los DataFrames nuevos (sin duplicados) junto a los otros procesados
SALIDA = CARPETA_DATOS / "processed"
doc25_sin_dups.to_csv(SALIDA / "docentes_2025_dedup.csv", index=False)
doc26_sin_dups.to_csv(SALIDA / "docentes_2026_dedup.csv", index=False)
print("Guardados:")
print("  ", (SALIDA / "docentes_2025_dedup.csv").name, doc25_sin_dups.shape)
print("  ", (SALIDA / "docentes_2026_dedup.csv").name, doc26_sin_dups.shape)

Guardados:
   docentes_2025_dedup.csv (141923, 25)
   docentes_2026_dedup.csv (155104, 25)


## 5. Desfasajes después de limpiar duplicados (con porcentaje)

Repetimos la detección de desfasajes, pero ahora sobre `doc25_sin_dups` / `doc26_sin_dups` (ya sin filas duplicadas), y agregamos el **porcentaje** de valores desfasados por columna y el porcentaje de combinaciones con cantidad desfasada.

In [33]:
# Preparar dtypes sobre los datasets ya deduplicados y detectar desfasajes.
# con_filas=True -> la tabla de categorias ya trae n_filas; el dict trae ademas
# 'cantidades' y 'resumen', asi que compare_datasets se ejecuta UNA sola vez.
doc25_sin_dups_c, doc26_sin_dups_c = preparar_para_comparar(doc25_sin_dups, doc26_sin_dups)

desfasajes_dedup = detectar_desfasajes(
    doc25_sin_dups_c, doc26_sin_dups_c, name1="2025", name2="2026", con_filas=True
)
desfasajes_dedup["resumen"]

Columnas vacias en un dataset (excluidas): ['IVSMEDIA', 'ivsmedia_q', 'contexto_zona']
Columnas mixto (num/texto) pasadas a texto: (ninguna)


{'columnas_solo_en_2025': [],
 'columnas_solo_en_2026': [],
 'columnas_categoricas_comparadas': ['ID_persona',
  'ID_CENTRO_docentes',
  'Sexo',
  'ZONA',
  'tipo_centro',
  'Rubro',
  'dept_nombre',
  'ciclo',
  'grado',
  'grupo',
  'MATERIA',
  'CONTEXTO',
  'tipo'],
 'columnas_numericas_comparadas': ['Dias6',
  'Dias5',
  'Dias4',
  'ANIOLECTIVO',
  'contexto_q',
  'vuln_q',
  'dias_totales',
  'accedio',
  'anio'],
 'columnas_con_categorias_desfasadas': ['ID_persona',
  'ID_CENTRO_docentes',
  'ciclo',
  'grado',
  'grupo',
  'MATERIA'],
 'combinaciones_comunes_comparadas': 25907,
 'combinaciones_con_cantidad_desfasada': 158483,
 'filas_totales_2025': 141923,
 'filas_totales_2026': 155104}

In [34]:
# La deteccion ya viene con n_filas (con_filas=True); solo extraemos la tabla.
desfasajes_df = desfasajes_dedup["categorias"]

print("Filas del df de desfasajes:", len(desfasajes_df))
desfasajes_df.head(20)

Filas del df de desfasajes: 12971


,columna,valor,solo_en,n_filas
0,ID_CENTRO_docentes,centro_2513,2026,144
1,ID_CENTRO_docentes,centro_2511,2026,129
2,ID_CENTRO_docentes,centro_0366,2025,79
3,ID_CENTRO_docentes,centro_2510,2026,77
4,ID_CENTRO_docentes,centro_2512,2026,70
5,ID_CENTRO_docentes,centro_2514,2026,67
6,ID_CENTRO_docentes,centro_2556,2026,51
7,ID_CENTRO_docentes,centro_2524,2026,47
8,ID_CENTRO_docentes,centro_2536,2026,42
9,ID_CENTRO_docentes,centro_2535,2026,36


In [35]:
# % de categorias desfasadas por columna, en DOS bases:
#   - %_valores : sobre el total de valores unicos (union 2025+2026) de la columna
#   - %_filas   : sobre el total de filas (2025 + 2026 deduplicados)
filas_2025 = len(doc25_sin_dups_c)
filas_2026 = len(doc26_sin_dups_c)
total_filas = filas_2025 + filas_2026

filas_pct = []
for col in desfasajes_df["columna"].unique():
    sub = desfasajes_df[desfasajes_df["columna"] == col]
    total_unicos = len(
        set(doc25_sin_dups_c[col].dropna().unique()) | set(doc26_sin_dups_c[col].dropna().unique())
    )
    n_valores = len(sub)
    n_filas_desf = int(sub["n_filas"].sum())
    filas_pct.append({
        "columna": col,
        "valores_desfasados": n_valores,
        "valores_unicos_totales": total_unicos,
        "%_valores": round(100 * n_valores / total_unicos, 2) if total_unicos else 0.0,
        "filas_desfasadas": n_filas_desf,
        "filas_totales": total_filas,
        "%_filas": round(100 * n_filas_desf / total_filas, 2),
    })
pct_categorias = pd.DataFrame(filas_pct).sort_values("%_filas", ascending=False)
pct_categorias

,columna,valores_desfasados,valores_unicos_totales,%_valores,filas_desfasadas,filas_totales,%_filas
2,MATERIA,311,1124,27.67,28600,297027,9.63
1,ID_persona,12253,43694,28.04,26709,297027,8.99
5,grupo,211,664,31.78,23466,297027,7.90
3,ciclo,4,25,16.00,1971,297027,0.66
0,ID_CENTRO_docentes,191,2644,7.22,1718,297027,0.58
4,grado,1,15,6.67,800,297027,0.27


In [36]:
# % de combinaciones con AL MENOS una cantidad desfasada, sobre las combinaciones comunes.
# Ojo: 'cantidades' trae una fila POR CADA columna numerica desfasada
# (una misma combinacion puede aparecer varias veces, una por Dias4/Dias5/...),
# asi que contamos combinaciones UNICAS, no el total de filas.
resumen = desfasajes_dedup["resumen"]
key_cols = resumen["columnas_categoricas_comparadas"]
combinaciones_unicas_desfasadas = desfasajes_dedup["cantidades"][key_cols].drop_duplicates().shape[0]

comunes = resumen["combinaciones_comunes_comparadas"]
pct_cantidades = round(100 * combinaciones_unicas_desfasadas / comunes, 2) if comunes else 0.0

print(f"Combinaciones comunes comparadas:                  {comunes}")
print(f"Filas de cantidad desfasada (1 por columna numerica): {resumen['combinaciones_con_cantidad_desfasada']}")
print(f"Combinaciones UNICAS con al menos 1 cantidad desfasada: {combinaciones_unicas_desfasadas}")
print(f"% de combinaciones con cantidad desfasada: {pct_cantidades}%")

Combinaciones comunes comparadas:                  25907
Filas de cantidad desfasada (1 por columna numerica): 158483
Combinaciones UNICAS con al menos 1 cantidad desfasada: 25907
% de combinaciones con cantidad desfasada: 100.0%


In [37]:
# Top valores desfasados por cantidad de filas, para CADA columna desfasada
# (mismo desglose que se veia para ID_CENTRO_docentes, ahora por columna).
TOP = 15
for col in sorted(desfasajes_df["columna"].unique()):
    sub = desfasajes_df[desfasajes_df["columna"] == col]
    print(f"\n=== {col} — {len(sub)} valores desfasados, {int(sub['n_filas'].sum())} filas ===")
    display(sub.head(TOP).reset_index(drop=True))

print(doc25_sin_dups_c['grado'].unique())


=== ID_CENTRO_docentes — 191 valores desfasados, 1718 filas ===


,columna,valor,solo_en,n_filas
0,ID_CENTRO_docentes,centro_2513,2026,144
1,ID_CENTRO_docentes,centro_2511,2026,129
2,ID_CENTRO_docentes,centro_0366,2025,79
3,ID_CENTRO_docentes,centro_2510,2026,77
4,ID_CENTRO_docentes,centro_2512,2026,70
5,ID_CENTRO_docentes,centro_2514,2026,67
6,ID_CENTRO_docentes,centro_2556,2026,51
7,ID_CENTRO_docentes,centro_2524,2026,47
8,ID_CENTRO_docentes,centro_2536,2026,42
9,ID_CENTRO_docentes,centro_2535,2026,36



=== ID_persona — 12253 valores desfasados, 26709 filas ===


,columna,valor,solo_en,n_filas
0,ID_persona,id_40792,2026,21
1,ID_persona,id_42647,2026,21
2,ID_persona,id_42824,2026,19
3,ID_persona,id_37162,2026,17
4,ID_persona,id_41449,2026,17
5,ID_persona,id_35308,2025,16
6,ID_persona,id_38048,2026,16
7,ID_persona,id_32440,2025,15
8,ID_persona,id_37550,2026,15
9,ID_persona,id_05436,2025,14



=== MATERIA — 311 valores desfasados, 28600 filas ===


...


=== grado — 1 valores desfasados, 800 filas ===


,columna,valor,solo_en,n_filas
0,grado,no aplica,2026,800



=== grupo — 211 valores desfasados, 23466 filas ===


,columna,valor,solo_en,n_filas
0,grupo,2do.cv g. 1,2026,1689
1,grupo,3ro.hcp g. 1,2026,1640
2,grupo,3ro.cshd g. 1,2025,1628
3,grupo,sin grupo a cargo||0,2026,1608
4,grupo,3ro.ctq g. 1,2025,912
5,grupo,3ro.ct g. 1,2026,902
6,grupo,2do.cv g. 2,2026,900
7,grupo,3ro.hcp g. 2,2026,856
8,grupo,3ro.cshd g. 2,2025,757
9,grupo,ra1||1,2025,681


<ArrowStringArray>
[               '1',                '2',                '3',
                '7',                '8',               '4º',
                '9',               '3º',               '2º',
               '6º',               '1º',               '5º',
                '0', 'Seminario Praxis']
Length: 14, dtype: str


In [38]:
# DataFrame de desfasajes de categorias (solo se muestra, no se guarda)
print("desfasajes_df:", desfasajes_df.shape)
desfasajes_df.head(20)

desfasajes_df: (12971, 4)


,columna,valor,solo_en,n_filas
0,ID_CENTRO_docentes,centro_2513,2026,144
1,ID_CENTRO_docentes,centro_2511,2026,129
2,ID_CENTRO_docentes,centro_0366,2025,79
3,ID_CENTRO_docentes,centro_2510,2026,77
4,ID_CENTRO_docentes,centro_2512,2026,70
5,ID_CENTRO_docentes,centro_2514,2026,67
6,ID_CENTRO_docentes,centro_2556,2026,51
7,ID_CENTRO_docentes,centro_2524,2026,47
8,ID_CENTRO_docentes,centro_2536,2026,42
9,ID_CENTRO_docentes,centro_2535,2026,36
